# GuidaPlate — Hidden Markov Model Dietary Risk Analysis
## Notebook 07

**Student:** ISIMBI TUZINDE Jade Keslie
**Supervisor:** Emmanuel Adjei
**Institution:** African Leadership University | BSc Software Engineering
**Date:** June 2026

### Purpose
This notebook uses a Gaussian Hidden Markov Model (HMM) to:
1. Identify hidden dietary risk states from nutrient patterns
2. Learn transition probabilities between states across CKD disease progression
3. Align HMM-discovered states with KDOQI rule-based risk labels
4. Provide an unsupervised validation of the labeling strategy used to train XGBoost

### Why HMM complements XGBoost and LSTM
- XGBoost classifies daily dietary risk from 9 aggregate features (supervised)
- LSTM detects patterns across 6 meal occasions (supervised, sequential)
- HMM discovers dietary risk states directly from nutrient distributions without 
  being given the labels (unsupervised), then alignment with KDOQI labels 
  validates whether the data-driven states reflect clinical thresholds

### Sequence construction
Patients are sorted by CKD stage (G2 → G3a → G3b → G4) to form 4 sequences.
This allows the HMM to learn transitions between dietary patterns across the 
disease progression continuum — patients within the same stage share similar 
clinical constraints, and cross-stage transitions reflect how dietary behaviour 
changes as kidney function declines.

In [26]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

try:
    from hmmlearn import hmm
    print("hmmlearn OK")
except ImportError:
    print("Run: pip3 install hmmlearn")
    raise

FIGURES_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR   = ROOT / 'outputs' / 'stats'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print("All imports OK")

hmmlearn OK
ROOT: /Users/jade/GUIDAPLATE
All imports OK


In [27]:
# Load cohort and risk labels
cohort = pd.read_csv(ROOT / 'data' / 'processed' / 'ckd_cohort_final.csv')
labels = pd.read_csv(ROOT / 'outputs' / 'stats' / '05_risk_labels.csv')

# Merge on SEQN
df = cohort.merge(labels[['SEQN', 'risk_label']], on='SEQN', how='inner')

# Drop missing nutrients — HMM cannot handle NaN
NUTRIENTS = ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']
df_clean = df.dropna(subset=NUTRIENTS).reset_index(drop=True)

print(f"Patients before dropping NaN: {len(df)}")
print(f"Patients after dropping NaN:  {len(df_clean)}")
print(f"\nRisk label distribution:\n{df_clean['risk_label'].value_counts()}")
print(f"\nStage distribution:\n{df_clean['ckd_stage'].value_counts()}")
print(f"\nFeature matrix shape: {df_clean[NUTRIENTS].shape}")

Patients before dropping NaN: 1862
Patients after dropping NaN:  1476

Risk label distribution:
risk_label
HIGH        1014
LOW          265
MODERATE     197
Name: count, dtype: int64

Stage distribution:
ckd_stage
G2     1145
G3a     226
G3b      80
G4       25
Name: count, dtype: int64

Feature matrix shape: (1476, 4)


In [28]:
# Scale features using StandardScaler
scaler = StandardScaler()
X_all = df_clean[NUTRIENTS].values
X_scaled_all = scaler.fit_transform(X_all)

print("Features scaled with StandardScaler")
print(f"Mean after scaling (should be ~0): {X_scaled_all.mean(axis=0).round(4)}")
print(f"Std after scaling  (should be ~1): {X_scaled_all.std(axis=0).round(4)}")

Features scaled with StandardScaler
Mean after scaling (should be ~0): [ 0. -0.  0. -0.]
Std after scaling  (should be ~1): [1. 1. 1. 1.]


In [29]:
# Sort patients by CKD stage to construct meaningful sequences
# G2 → G3a → G3b → G4 represents disease progression direction
# Each stage becomes one sequence — HMM learns transitions
# between dietary patterns as disease progresses

STAGE_ORDER_MAP = {'G2': 1, 'G3a': 2, 'G3b': 3, 'G4': 4}
df_clean['stage_order'] = df_clean['ckd_stage'].map(STAGE_ORDER_MAP)
df_sorted = df_clean.sort_values(
    ['stage_order', 'risk_label']
).reset_index(drop=True)

X_sorted = df_sorted[NUTRIENTS].values
X_sorted_scaled = scaler.transform(X_sorted)

# Build sequence lengths — one sequence per CKD stage
stage_lengths = []
print("Sequence construction:")
for stage in ['G2', 'G3a', 'G3b', 'G4']:
    count = int((df_sorted['ckd_stage'] == stage).sum())
    if count > 0:
        stage_lengths.append(count)
        print(f"  Stage {stage}: {count} patients")

print(f"\nTotal observations: {sum(stage_lengths)}")
print(f"Number of sequences: {len(stage_lengths)}")
print(f"Sequence lengths: {stage_lengths}")

N_STATES    = 4
N_ITER      = 200
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

hmm_model = hmm.GaussianHMM(
    n_components=N_STATES,
    covariance_type='full',
    n_iter=N_ITER,
    random_state=RANDOM_SEED,
    tol=1e-4
)

hmm_model.fit(X_sorted_scaled, stage_lengths)

print(f"\nHMM training converged: {hmm_model.monitor_.converged}")
print(f"Final log-likelihood:   {hmm_model.monitor_.history[-1]:.4f}")
print(f"\nInitial state probabilities (π):")
for i, p in enumerate(hmm_model.startprob_):
    print(f"  State {i}: {p:.4f}")

Sequence construction:
  Stage G2: 1145 patients
  Stage G3a: 226 patients
  Stage G3b: 80 patients
  Stage G4: 25 patients

Total observations: 1476
Number of sequences: 4
Sequence lengths: [1145, 226, 80, 25]

HMM training converged: True
Final log-likelihood:   -4878.4981

Initial state probabilities (π):
  State 0: 0.0000
  State 1: 0.5999
  State 2: 0.4001
  State 3: 0.0000


In [ ]:
# ── Model selection — BIC score for 2, 3, 4 hidden states ──
from hmmlearn import hmm as hmmlib
import numpy as np

print("BIC Model Selection — testing 2, 3, 4 hidden states")
print("=" * 52)

bic_scores = {}
ll_scores  = {}

for n in [2, 3, 4]:
    test_model = hmmlib.GaussianHMM(
        n_components=n,
        covariance_type='full',
        n_iter=200,
        random_state=42
    )
    test_model.fit(X_sorted_scaled, stage_lengths)
    log_likelihood = test_model.score(
        X_sorted_scaled, stage_lengths
    )
    # BIC = -2 * logL + n_params * log(n_samples)
    # n_params = transitions (n²) + means (n×4) + 
    #            covariance diag (n×4)
    n_params = (n ** 2) + (n * 4) + (n * 4)
    n_samples = len(X_sorted_scaled)
    bic = -2 * log_likelihood + n_params * np.log(n_samples)
    bic_scores[n] = round(bic, 2)
    ll_scores[n]  = round(log_likelihood, 2)
    print(f"  States: {n} | "
          f"Log-likelihood: {log_likelihood:10.2f} | "
          f"n_params: {n_params:3d} | "
          f"BIC: {bic:.2f}")

best_n = min(bic_scores, key=bic_scores.get)
print("=" * 52)
print(f"Best number of states by BIC: {best_n}")
print(f"BIC scores: {bic_scores}")
print()
if best_n == 3:
    print("Current choice of 3 states is optimal.")
elif best_n == 2:
    print("Data supports 2 states — LOW and HIGH are the")
    print("natural clusters. MODERATE is a transitional zone.")
else:
    print(f"Consider rerunning Cell 5 with N_STATES = {best_n}")

# Save BIC results
bic_df = pd.DataFrame({
    'n_states': list(bic_scores.keys()),
    'log_likelihood': list(ll_scores.values()),
    'bic_score': list(bic_scores.values())
})
bic_df['optimal'] = bic_df['n_states'] == best_n
bic_df.to_csv(
    STATS_DIR / '14_hmm_bic_model_selection.csv',
    index=False
)
print("\nSaved: 14_hmm_bic_model_selection.csv")

BIC Model Selection — testing 2, 3, 4 hidden states
  States: 2 | Log-likelihood:   -5335.68 | n_params:  20 | BIC: 10817.30
  States: 3 | Log-likelihood:   -4977.28 | n_params:  33 | BIC: 10195.36
  States: 4 | Log-likelihood:   -4878.56 | n_params:  48 | BIC: 10107.38
Best number of states by BIC: 4
BIC scores: {2: 10817.3, 3: 10195.36, 4: 10107.38}

Consider rerunning Cell 5 with N_STATES = 4

Saved: 14_hmm_bic_model_selection.csv


In [31]:
# Decode hidden states for all patients
hidden_states = hmm_model.predict(X_sorted_scaled, stage_lengths)

df_sorted = df_sorted.copy()
df_sorted['hmm_state'] = hidden_states

# Use df_sorted as df_clean for all remaining cells
df_clean = df_sorted.copy()

print("Hidden state distribution:")
for state in range(N_STATES):
    count = int((hidden_states == state).sum())
    print(f"  State {state}: {count} patients "
          f"({count / len(hidden_states) * 100:.1f}%)")

Hidden state distribution:
  State 0: 462 patients (31.3%)
  State 1: 446 patients (30.2%)
  State 2: 487 patients (33.0%)
  State 3: 81 patients (5.5%)


In [32]:
# Align HMM states (arbitrary integers) with KDOQI labels
# Each HMM state is mapped to the KDOQI label it most overlaps with

print("HMM state vs KDOQI label cross-tabulation:")
crosstab = pd.crosstab(
    df_clean['hmm_state'],
    df_clean['risk_label'],
    margins=True
)
print(crosstab)

state_label_map = {}
for state in range(N_STATES):
    state_patients = df_clean[df_clean['hmm_state'] == state]['risk_label']
    if len(state_patients) > 0:
        dominant_label = state_patients.value_counts().index[0]
        state_label_map[state] = dominant_label
        pct = (state_patients.value_counts().iloc[0] /
               len(state_patients) * 100)
        print(f"\nState {state} → {dominant_label} "
              f"({state_patients.value_counts().iloc[0]} / "
              f"{len(state_patients)} patients, {pct:.1f}%)")

df_clean['hmm_label'] = df_clean['hmm_state'].map(state_label_map)
print(f"\nFinal HMM label distribution:\n{df_clean['hmm_label'].value_counts()}")

HMM state vs KDOQI label cross-tabulation:
risk_label  HIGH  LOW  MODERATE   All
hmm_state                            
0              0  265       197   462
1            446    0         0   446
2            487    0         0   487
3             81    0         0    81
All         1014  265       197  1476

State 0 → LOW (265 / 462 patients, 57.4%)

State 1 → HIGH (446 / 446 patients, 100.0%)

State 2 → HIGH (487 / 487 patients, 100.0%)

State 3 → HIGH (81 / 81 patients, 100.0%)

Final HMM label distribution:
hmm_label
HIGH    1014
LOW      462
Name: count, dtype: int64


In [33]:
# Figure 19 — Transition matrix heatmap
state_names = [state_label_map.get(i, f'State {i}')
               for i in range(N_STATES)]

transition_df = pd.DataFrame(
    hmm_model.transmat_,
    index=state_names,
    columns=state_names
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    transition_df,
    annot=True, fmt='.3f',
    cmap='Blues', vmin=0, vmax=1,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Transition Probability'}
)
ax.set_title(
    'HMM Dietary Risk State Transition Matrix\n'
    'GuidaPlate — NHANES 2017-2018 CKD Cohort',
    fontsize=12, pad=12
)
ax.set_xlabel('To State', fontsize=11)
ax.set_ylabel('From State', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '19_hmm_transition_matrix.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 19_hmm_transition_matrix.png")

transition_df.to_csv(STATS_DIR / '11_hmm_transition_probabilities.csv')
print("Saved: 11_hmm_transition_probabilities.csv")

print("\nTransition matrix:")
print(transition_df.round(3).to_string())

Saved: 19_hmm_transition_matrix.png
Saved: 11_hmm_transition_probabilities.csv

Transition matrix:
        LOW   HIGH   HIGH   HIGH
LOW   1.000  0.000  0.000  0.000
HIGH  0.003  0.443  0.441  0.113
HIGH  0.006  0.401  0.486  0.106
HIGH  0.000  0.670  0.330  0.000


In [34]:
# Figure 20 — HMM state nutrient profiles
state_means_original = scaler.inverse_transform(hmm_model.means_)
nutrient_labels = ['Potassium\n(mg/day)', 'Phosphorus\n(mg/day)',
                   'Protein\n(g/kg/day)', 'Sodium\n(mg/day)']
colors = {'LOW': '#2E86AB', 'MODERATE': '#F4A261', 'HIGH': '#E63946'}

fig, axes = plt.subplots(1, N_STATES, figsize=(14, 5))

for state in range(N_STATES):
    label = state_label_map.get(state, f'State {state}')
    color = colors.get(label, '#888888')
    count = int((hidden_states == state).sum())
    axes[state].bar(
        nutrient_labels,
        state_means_original[state],
        color=color, alpha=0.8, edgecolor='white'
    )
    axes[state].set_title(
        f'State {state}: {label}\n({count} patients)',
        fontsize=11, fontweight='bold', color=color
    )
    axes[state].set_ylabel('Mean Daily Intake', fontsize=9)
    axes[state].tick_params(axis='x', labelsize=8)
    for i, val in enumerate(state_means_original[state]):
        axes[state].text(
            i, val + abs(val) * 0.02,
            f'{val:.0f}',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle(
    'HMM Hidden State Nutrient Profiles\n'
    'Mean daily nutrient intake per dietary risk state',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '20_hmm_state_profiles.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 20_hmm_state_profiles.png")

Saved: 20_hmm_state_profiles.png


In [35]:
# Figure 21 — HMM vs KDOQI confusion matrix
valid_mask = df_clean['hmm_label'].notna()
hmm_pred   = df_clean.loc[valid_mask, 'hmm_label']
kdoqi_true = df_clean.loc[valid_mask, 'risk_label']

agreement = accuracy_score(kdoqi_true, hmm_pred)
print(f"HMM vs KDOQI agreement: {agreement:.1%}")
print(f"\nClassification report:")
print(classification_report(kdoqi_true, hmm_pred,
                             target_names=['LOW', 'MODERATE', 'HIGH'],
                             zero_division=0))

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(
    kdoqi_true, hmm_pred,
    labels=['LOW', 'MODERATE', 'HIGH']
)
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=['LOW', 'MODERATE', 'HIGH'],
    yticklabels=['LOW', 'MODERATE', 'HIGH'],
    cmap='Blues', ax=ax
)
ax.set_title(
    f'HMM Discovered States vs KDOQI Rule-Based Labels\n'
    f'Agreement: {agreement:.1%}',
    fontsize=12
)
ax.set_xlabel('HMM Predicted State', fontsize=11)
ax.set_ylabel('KDOQI Rule-Based Label', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '21_hmm_vs_kdoqi.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 21_hmm_vs_kdoqi.png")

HMM vs KDOQI agreement: 86.7%

Classification report:
              precision    recall  f1-score   support

         LOW       1.00      1.00      1.00      1014
    MODERATE       0.57      1.00      0.73       265
        HIGH       0.00      0.00      0.00       197

    accuracy                           0.87      1476
   macro avg       0.52      0.67      0.58      1476
weighted avg       0.79      0.87      0.82      1476

Saved: 21_hmm_vs_kdoqi.png


In [36]:
# Figure 22 — HMM state distribution by CKD stage
stage_state = pd.crosstab(
    df_clean['ckd_stage'],
    df_clean['hmm_label'],
    normalize='index'
) * 100

stage_order = ['G2', 'G3a', 'G3b', 'G4']
label_order = ['LOW', 'MODERATE', 'HIGH']
plot_colors = ['#2E86AB', '#F4A261', '#E63946']

stage_state = stage_state.reindex(
    index=[s for s in stage_order if s in stage_state.index],
    columns=[l for l in label_order if l in stage_state.columns],
    fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 6))
stage_state.plot(
    kind='bar', stacked=True,
    color=plot_colors, ax=ax,
    edgecolor='white', linewidth=0.5
)
ax.set_title(
    'HMM Dietary Risk State Distribution by CKD Stage\n'
    'Percentage of patients per state within each stage',
    fontsize=12
)
ax.set_xlabel('CKD Stage', fontsize=11)
ax.set_ylabel('Percentage of Patients (%)', fontsize=11)
ax.legend(title='HMM State', bbox_to_anchor=(1.05, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_hmm_state_by_stage.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 22_hmm_state_by_stage.png")

Saved: 22_hmm_state_by_stage.png


In [37]:
# Save all results to outputs/stats/
hmm_results = df_clean[[
    'SEQN', 'ckd_stage', 'potassium', 'phosphorus',
    'protein_per_kg', 'sodium', 'risk_label',
    'hmm_state', 'hmm_label'
]].copy()

hmm_results.to_csv(
    STATS_DIR / '12_hmm_patient_states.csv',
    index=False
)
print("Saved: 12_hmm_patient_states.csv")

summary = df_clean.groupby('hmm_label')[NUTRIENTS].mean().round(2)
summary.to_csv(STATS_DIR / '13_hmm_state_means.csv')
print("Saved: 13_hmm_state_means.csv")

Saved: 12_hmm_patient_states.csv
Saved: 13_hmm_state_means.csv


In [38]:
print("=" * 50)
print("NOTEBOOK 07 COMPLETE")
print("=" * 50)
print(f"\nHMM trained on {len(df_clean)} patients")
print(f"Hidden states: {N_STATES} (aligned to LOW / MODERATE / HIGH)")
print(f"HMM vs KDOQI agreement: {agreement:.1%}")
print(f"\nSequence construction: 4 stage-grouped sequences")
print(f"  G2: {stage_lengths[0]} | G3a: {stage_lengths[1]} | "
      f"G3b: {stage_lengths[2]} | G4: {stage_lengths[3]}")
print("\nSaved figures:")
for f in ['19_hmm_transition_matrix.png', '20_hmm_state_profiles.png',
          '21_hmm_vs_kdoqi.png', '22_hmm_state_by_stage.png']:
    print(f"  outputs/figures/{f}")
print("\nSaved stats:")
for f in ['11_hmm_transition_probabilities.csv',
          '12_hmm_patient_states.csv',
          '13_hmm_state_means.csv']:
    print(f"  outputs/stats/{f}")
print("\nNext: backend integration (July 2026)")

NOTEBOOK 07 COMPLETE

HMM trained on 1476 patients
Hidden states: 4 (aligned to LOW / MODERATE / HIGH)
HMM vs KDOQI agreement: 86.7%

Sequence construction: 4 stage-grouped sequences
  G2: 1145 | G3a: 226 | G3b: 80 | G4: 25

Saved figures:
  outputs/figures/19_hmm_transition_matrix.png
  outputs/figures/20_hmm_state_profiles.png
  outputs/figures/21_hmm_vs_kdoqi.png
  outputs/figures/22_hmm_state_by_stage.png

Saved stats:
  outputs/stats/11_hmm_transition_probabilities.csv
  outputs/stats/12_hmm_patient_states.csv
  outputs/stats/13_hmm_state_means.csv

Next: backend integration (July 2026)
